[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C42_Learning_Theory_Course/00_setup/00_environment_check.ipynb)

# 00 · 环境自检与三个引子实验

本 notebook 做两件事：
1. **环境自检**：确认 numpy 可用、版本、随机种子可复现。本课全程纯 numpy / CPU。
2. **三个引子实验**：用最小代码先「看见」后面三个模块的核心现象——经验风险随样本收敛（M01）、GD 的收敛速率（M02）、过参数化能拟合随机标签（M05）。

> 本课的信条：**理论是可被实验反复确认的预测**。先在这里建立这个肌肉记忆。

## 1 · 环境自检

In [ ]:
import sys, platform
import numpy as np
print('Python  :', sys.version.split()[0])
print('platform:', platform.system(), platform.machine())
print('numpy   :', np.__version__)
rng = np.random.default_rng(0)
# 复现性：同一种子两次抽样必须一致
a = np.random.default_rng(42).standard_normal(5)
b = np.random.default_rng(42).standard_normal(5)
assert np.allclose(a, b), '同种子应可复现'
print('\n✅ 环境就绪；随机种子可复现（本课所有实验固定种子，结果可对拍）')

## 2 · 引子一（→M01）：经验风险随样本量收敛到真实风险

固定一个假设，它的**真实风险**是某个常数 $p$（这里用一枚偏置硬币：损失=1 的概率为 $p$）。
我们抽 $n$ 个样本算**经验风险**，看它随 $n$ 增大如何收敛到 $p$，且偏差以 $\sim 1/\sqrt n$ 缩小。
这是 Hoeffding 不等式说的事——模块 01 会把它升级成对**整个假设类**的一致控制。

In [ ]:
p_true = 0.3                      # 这个固定假设的真实风险
rng = np.random.default_rng(0)
print(f'{"n":>7s}  {"经验风险":>8s}  {"|偏差|":>8s}  {"1/sqrt(n)":>10s}')
for n in [10, 100, 1000, 10000, 100000]:
    # 经验风险 = n 个 Bernoulli(p) 损失的平均
    losses = (rng.random(n) < p_true).astype(float)
    emp = losses.mean()
    print(f'{n:>7d}  {emp:>8.4f}  {abs(emp-p_true):>8.4f}  {1/np.sqrt(n):>10.4f}')
# 偏差应随 n 单调（统计上）缩小，且与 1/sqrt(n) 同量级
big = (rng.random(100000) < p_true).mean()
assert abs(big - p_true) < 0.01, '大样本下经验风险应非常接近真实风险'
print('\n✅ 经验风险 → 真实风险，偏差 ~ 1/sqrt(n)。这是泛化界的最底层砖块。')

## 3 · 引子二（→M02）：梯度下降的收敛速率

在一个**凸**目标 $f(x)=\tfrac12 x^\top A x$（$A$ 正定，最优在 0，$f^\*=0$）上跑 GD，步长 $1/L$（$L$ 为最大特征值）。
理论（模块 02）：$f(x_t)-f^\* \le \dfrac{L\|x_0-x^\*\|^2}{2t}$，即 $O(1/t)$。我们实测曲线并验证它**确实**压在这条上界之下。

In [ ]:
rng = np.random.default_rng(1)
d = 10
B = rng.standard_normal((d, d)); A = B @ B.T / d + 0.1*np.eye(d)   # 对称正定
L = np.linalg.eigvalsh(A).max()                                    # 光滑常数 = 最大特征值
x0 = rng.standard_normal(d)
f   = lambda x: 0.5 * x @ A @ x
grad= lambda x: A @ x
x = x0.copy(); T = 500; fs = []
for t in range(T):
    x = x - (1.0/L) * grad(x)
    fs.append(f(x))
fs = np.array(fs)
ub = L * np.dot(x0, x0) / (2 * np.arange(1, T+1))                  # O(1/t) 理论上界
for t in [1, 10, 100, 500]:
    print(f't={t:4d}  f(x_t)={fs[t-1]:.3e}   O(1/t)上界={ub[t-1]:.3e}   实测<=上界? {fs[t-1] <= ub[t-1]}')
assert np.all(fs <= ub + 1e-12), 'GD 凸光滑 O(1/t) 上界必须对所有 t 成立'
print('\n✅ 实测损失逐点压在 L||x0-x*||^2/(2t) 之下 —— O(1/t) 收敛率被钉死。')

## 4 · 引子三（→M05）：过参数化能拟合**随机标签**

这是 Zhang et al. 2017 引爆当代泛化研究的实验的最小版：当参数维度 $d$ ≥ 样本数 $n$ 时，一个**线性**模型就能把**任意**标签（包括纯随机的）拟合到零误差。
含义：模型的表达容量足以**记忆**任意数据，所以「能拟合训练集」本身不解释泛化——经典的『容量 ⇒ 泛化』链条在过参数化下断裂。模块 05 全程处理这个难题。

In [ ]:
rng = np.random.default_rng(2)
n, d = 30, 60                                  # 过参数化：d > n
X = rng.standard_normal((n, d))
y_random = rng.choice([-1.0, 1.0], size=n)     # 纯随机标签！与 X 无任何关系
# 最小范数插值解：w = X^T (X X^T)^{-1} y  （n<=d 时存在且插值）
w = X.T @ np.linalg.solve(X @ X.T, y_random)
pred = X @ w
train_err = np.mean(np.sign(pred) != y_random)
print(f'拟合随机标签的训练误差 = {train_err:.4f}   残差范数 = {np.linalg.norm(pred - y_random):.2e}')
assert np.linalg.norm(pred - y_random) < 1e-6, '过参数化(d>=n)下应能精确拟合任意标签'
print('✅ d>n 时连纯随机标签都能零误差拟合 —— 「能记忆」≠「会泛化」。')
print('   这正是 M05 要回答的：那为什么真实数据上训练出的网络又泛化得好？')

---
### 小结

- 环境就绪：纯 numpy / CPU，固定种子可复现。
- **M01 预告**：经验风险以 $\sim1/\sqrt n$ 收敛到真实风险——泛化界的地基。
- **M02 预告**：凸光滑上 GD 实测损失精确压在 $O(1/t)$ 理论上界下。
- **M05 预告**：过参数化能拟合随机标签，经典容量界因此失效。

下一站：**模块 01 · 统计学习理论**——把单个假设的 Hoeffding 集中，升级成对整个假设类的一致收敛，并用 VC 维 / Rademacher 复杂度度量「类有多丰富」。